# Shape Assembly

In [ ]:
import pyglet
pyglet.options["headless"] = True

import numpy as np



import warp as wp
import torch
import pyvista as pv
from waxmorph.render import PyVistaInterface

pv.set_jupyter_backend("html")

wp.init()
device = "cuda"
torch_device = torch.device("cuda")

np.random.seed(42)
torch.manual_seed(42)

## 1. Load meshes and sample point clouds

In [ ]:
from waxmorph.data import sample_mesh_pair

N_POINTS = 2000
NUM_GENES = 32

data = sample_mesh_pair(
    source_path="meshes/bunny.ply",
    target_path="meshes/armadillo.ply",
    n_points=N_POINTS,
    source_extent=10.0,
    target_extent=14.29,
)

print(f"Source: {data['source_pos'].shape}, Target: {data['target_pos'].shape}")

## 2. Initialize state arrays

In [ ]:
N = len(data["source_pos"])

# Positions: start from armadillo
pos_np = data["source_pos"].copy()

# Radii: uniform
rad_np = np.full(N, data["source_radius"], dtype=np.float32)

# Polarities: random unit vectors
rng = np.random.default_rng(42)
pol_np = rng.standard_normal((N, 3)).astype(np.float32)
pol_np /= np.linalg.norm(pol_np, axis=-1, keepdims=True) + 1e-9

# Genes: random initialization
genes_np = rng.random((N, NUM_GENES)).astype(np.float32)

print(f"Initialized {N} source particles, {len(data['target_pos'])} target points, {NUM_GENES} genes")

## 2b. Render initial states (PyVista)

In [ ]:
morph_source = np.full(N, 0.5, dtype=np.float32)

plotter = PyVistaInterface.draw_3d_view(
    data["source_pos"], rad_np, morph_source, pol_np,
    particle_count=N, blim=-6, tlim=6, alpha=0.9,
    show_polarities=False,
)
plotter.add_title("Source (Armadillo)")
plotter.export_html("test.html")
plotter.show()

In [ ]:
N_target = len(data["target_pos"])
morph_target = np.full(N_target, 0.5, dtype=np.float32)
rad_target = np.full(N_target, data["target_radius"], dtype=np.float32)
pol_target = np.zeros((N_target, 3), dtype=np.float32)
pol_target[:, 2] = 1.0

plotter = PyVistaInterface.draw_3d_view(
    data["target_pos"], rad_target, morph_target, pol_target,
    particle_count=N_target, blim=-6, tlim=6, alpha=0.9,
    show_polarities=False,
)
plotter.add_title("Target (Bunny)")
plotter.show()

## 3. Build GNS model

In [ ]:
from waxmorph.graph import build_graph
from waxmorph.gnn import GNS
from waxmorph.losses import make_samples_loss

# Probe feature dimensions
X_probe = wp.from_numpy(pos_np.copy(), dtype=wp.vec3f, device=device)
P_probe = wp.from_numpy(pol_np.copy(), dtype=wp.vec3f, device=device)
R_probe = wp.from_numpy(rad_np.copy(), dtype=wp.float32, device=device)
G_probe = wp.from_numpy(genes_np.copy(), dtype=wp.float32, device=device)

node_feats, edge_index, edge_feats = build_graph(X_probe, P_probe, R_probe, particle_count=N, G=G_probe)

print(f"Node features: {node_feats.shape}")
print(f"Edge index: {edge_index.shape}  ({edge_index.shape[1]} directed edges)")
print(f"Edge features: {edge_feats.shape}")

model = GNS(
    node_feature_dim=node_feats.shape[1],
    edge_feature_dim=edge_feats.shape[1],
    node_latent_dim=256,
    edge_latent_dim=256,
    hidden_dim=256,
    num_mp_steps=5,
    num_mlp_layers=3,
    output_dims={"dX": 3, "dP": 3, "dG": NUM_GENES},
    checkpoint_processor=True,
).to(torch_device)

n_params = sum(p.numel() for p in model.parameters())
print(f"GNS parameters: {n_params:,}")

loss_fn = make_samples_loss(loss="sinkhorn")
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

## 4. Training

In [ ]:
from waxmorph.train import train, TrainConfig

config = TrainConfig(n_epochs=2000, t_rollout=100)
result = train(
    model, optimizer, loss_fn,
    source_pos=data["source_pos"],
    target_pos=data["target_pos"],
    polarities=pol_np, genes=genes_np, radii=rad_np,
    config=config,
    save_path="runs/bunny_armadillo_v3_256.pt",
)
model = result.model
losses = result.log["losses_total"]

## 5. Visualize loss curve

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, key, title in zip(axes, ["losses_shape", "losses_l2", "losses_total"],
                          ["Shape", "L2", "Total"]):
    ax.plot(result.log[key])
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title(f"Armadillo \u2192 Bunny ({title})")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Final rollout with trajectory capture

In [ ]:
trajectory = result.log["best_traj_pos"]
X_final = trajectory[-1]
print(f"Best trajectory: {trajectory.shape[0]} frames, best loss={result.log['best_loss']:.4f}")

## 7. Render final state (PyVista)

In [ ]:
morph_final = np.full(N, 0.5, dtype=np.float32)

plotter = PyVistaInterface.draw_3d_view(
    X_final, rad_np, morph_final, pol_np,
    particle_count=N, blim=-6, tlim=6, alpha=0.9,
    show_polarities=False,
)
plotter.add_title("Predicted (Final)")
plotter.show()

## 8. Deformation movie (WarpMovieRenderer)

In [ ]:
import os
from tqdm import trange
from waxmorph.render import WarpMovieRenderer

os.makedirs("Output", exist_ok=True)
VIDEO_PATH = "Output/shape_assembly_v3.usd"

INTERP_STEPS = 4
colors_rgb = np.full((N, 3), [0.3, 0.6, 0.9], dtype=np.float32)

with WarpMovieRenderer(
    filename=VIDEO_PATH,
    max_particles=N,
    width=1920,
    height=1080,
    fps=30,
    device="cuda",
) as mov:
    frame_idx = 0
    n_frames = trajectory.shape[0]
    for seg in trange(n_frames - 1):
        start = trajectory[seg]
        end = trajectory[seg + 1]
        for sub in range(INTERP_STEPS):
            alpha = sub / INTERP_STEPS
            s = 3.0 * alpha**2 - 2.0 * alpha**3
            pts = start * (1.0 - s) + end * s
            mov.write_frame_from_numpy(
                t=float(frame_idx),
                centers=pts,
                radii=rad_np,
                colors=colors_rgb,
                particle_count=N,
            )
            frame_idx += 1
    for _ in trange(15):
        mov.write_frame_from_numpy(
            t=float(frame_idx),
            centers=trajectory[-1],
            radii=rad_np,
            colors=colors_rgb,
            particle_count=N,
        )
        frame_idx += 1